<a href="https://colab.research.google.com/github/sparshbansal-newton/deep-learning-labs/blob/main/Notebooks/7_pytorch_training_pipeline/lab_pytorch_nn_module_breast_cancer_PRACTICE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5 (Practice): Backprop & Autograd inside a Real Training Loop
## Breast Cancer Diagnosis Classifier with PyTorch's `nn.Module`

### How to use this notebook

You've already worked through the **PyTorch Autograd** lab — you saw `requires_grad`, `.backward()`,
the `grad_fn` paper-trail, gradient **accumulation**, and `torch.no_grad()` on tiny scalar examples.

This notebook puts those exact ideas to work inside a **real classifier**. The data-loading and
preprocessing steps are done for you — they are *not* the point. Your job is to fill in the cells
marked **`TODO`**, which are the ones where autograd and backpropagation actually happen:

1. **`forward()`** — the pass that *builds* the autograd computation graph.
2. **The training loop** — forward → loss → `zero_grad` → `backward` → `step`.
3. **Evaluation under `torch.no_grad()`** — running the graph *without* recording it.

Each TODO has a **Hint** box right above it. Try to write the code from the concept first, and only
expand your memory of the autograd lab when you're stuck. Run every cell in order.

> **The one question to keep asking yourself:** *for each line I write, is a gradient being recorded
> here, or not?* That single question is what this whole lab is about.

---

## Recap: the 4 autograd facts you'll need

Straight from the autograd lab — keep these next to you:

| # | Fact | Where it bites in this lab |
|---|------|----------------------------|
| 1 | Operations on tensors with `requires_grad=True` are **recorded** into a graph (`grad_fn`). | `nn.Linear` weights have `requires_grad=True` by default, so your `forward()` builds the graph automatically. |
| 2 | `loss.backward()` walks that graph in reverse and fills each parameter's `.grad`. | The `backward()` call in the training loop. |
| 3 | Gradients **accumulate** — `.grad` is *added to*, not overwritten. | You **must** call `optimizer.zero_grad()` every epoch or gradients from old epochs leak in. |
| 4 | `torch.no_grad()` stops the graph from being built at all. | Used at evaluation, where you only need a forward pass. |

### The math (same as the scalar BCE example you did by hand)

$$z = w^\top x + b \qquad \hat{y} = \sigma(z) = \frac{1}{1+e^{-z}} \qquad L = -\frac{1}{N}\sum_i\big[y_i\log\hat y_i + (1-y_i)\log(1-\hat y_i)\big]$$

In the autograd lab you differentiated this **by hand** for one neuron and matched `w.grad`. Here the
same chain rule runs automatically over all 30 weights + bias — that's the only difference.

SGD update after `backward()`: $\;\theta \leftarrow \theta - \eta\,\nabla_\theta L$.

---

## Step 1: Imports (given)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

## Step 2: Data ingestion & preprocessing (given — not the focus)

The **Wisconsin Breast Cancer** dataset: 569 patients, 30 numeric features from a digitized image of a
breast mass, and a `diagnosis` label — `M` (malignant) or `B` (benign). Everything below is standard
and provided for you: drop junk columns → split → scale → encode → convert to `float32` tensors.

> Read it, run it, but you don't need to edit it. The autograd action starts in Step 3.

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

# hold out 20% BEFORE scaling/encoding so no test info leaks into preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2
)

# scale features to zero-mean/unit-variance (fit on train only)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# encode labels: B -> 0, M -> 1 (fit on train only)
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

# NumPy float64 -> PyTorch float32 (nn.Linear weights are float32; mismatch raises an error)
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

print("X_train_tensor:", X_train_tensor.shape)   # (455, 30)
print("y_train_tensor:", y_train_tensor.shape)   # (455,)  <- 1-D! remember this for the loss
print("requires_grad on the data?", X_train_tensor.requires_grad)  # False — data is not a parameter

---
## Step 3: The model — `TODO 1`: complete `forward()`

The classifier is logistic regression: one `nn.Linear(30, 1)` layer, then `nn.Sigmoid()` to turn the
raw score into a probability. The `__init__` is written for you.

> ### 💡 Hint — this is where the autograd graph gets built
> `forward()` just says *what* to compute. Because `self.linear`'s weights have `requires_grad=True`,
> every operation you write here is **recorded** — the returned tensor will carry a `grad_fn`, exactly
> like `y = x**2` did in the autograd lab. You need two lines:
> 1. push `features` through the linear layer → call it `out`
> 2. push `out` through the sigmoid → return it
>
> Call the layers you stored in `__init__`: `self.linear(...)` and `self.sigmoid(...)`.
> (Prefer `self.linear(x)` over `self.linear.forward(x)` — `__call__` also runs registered hooks.)

In [ ]:
class MySimpleNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()                          # MUST be first — sets up the nn.Module machinery
        self.linear = nn.Linear(num_features, 1)    # learnable: 30 weights + 1 bias (requires_grad=True)
        self.sigmoid = nn.Sigmoid()                 # stateless activation

    def forward(self, features):
        # features shape: (Batch_Size, num_features) -> return shape: (Batch_Size, 1)
        # ---- TODO 1: build the forward pass (2 lines) ----
        # out = ...            # linear layer applied to features
        # out = ...            # sigmoid applied to out
        # return out
        raise NotImplementedError("Complete TODO 1: the forward pass")

### Quick check that the graph is really being built

Run this after finishing TODO 1. If your `forward()` is right, the output tensor prints a
**`grad_fn=<SigmoidBackward0>`** — that attribute is autograd's proof it recorded your operations
(same paper-trail you watched appear in the autograd lab).

In [ ]:
_probe_model = MySimpleNN(X_train_tensor.shape[1])
_probe_out = _probe_model(X_train_tensor[:3])      # note: model(x), NOT model.forward(x)
print(_probe_out)
print("grad_fn present?", _probe_out.grad_fn is not None)   # should be True

### Hyperparameters & loss (given)

In [ ]:
learning_rate = 0.1
epochs = 25

# Binary Cross-Entropy — expects prediction and target to be the SAME shape, e.g. (N, 1)
loss_function = nn.BCELoss()

---
## Step 4: The training loop — `TODO 2` (the heart of this lab)

This is the whole point. Each epoch runs **five** steps. You've done every one of them individually in
the autograd lab — now you assemble them into a loop.

> ### 💡 Hint — the five steps, and *why each exists*
> | Step | Line to write | Autograd reason |
> |------|---------------|-----------------|
> | 1. **Forward** | `y_pred = model(X_train_tensor)` | builds the graph for this epoch |
> | 2. **Loss** | `loss = loss_function(y_pred, y_train_tensor.view(-1, 1))` | one scalar — `.backward()` only works on a scalar |
> | 3. **Zero grads** | `optimizer.zero_grad()` | gradients **accumulate**; clear last epoch's before this one (Fact #3) |
> | 4. **Backward** | `loss.backward()` | walk the graph, fill every parameter's `.grad` (Fact #2) |
> | 5. **Step** | `optimizer.step()` | apply $\theta \leftarrow \theta - \eta\nabla_\theta L$ using those `.grad`s |
>
> **Two traps carried over from the autograd lab:**
> - **`.view(-1, 1)` on the target.** `y_pred` is `(N, 1)` but `y_train_tensor` is `(N,)`. If you don't
>   reshape, PyTorch *broadcasts* `(N,)` vs `(N,1)` into `(N, N)` — no error, just a garbage loss.
> - **`zero_grad()` must come before `backward()`.** Skip it and epoch 2's gradient is epoch-1 + epoch-2
>   added together — the exact accumulation behavior you saw when you called `.backward()` twice without
>   `.zero_()` in the autograd lab.
>
> The optimizer is created for you. `model.parameters()` auto-discovers the weights & bias registered
> inside `MySimpleNN` — you never collect them by hand.

In [ ]:
# fresh model + optimizer
model = MySimpleNN(X_train_tensor.shape[1])
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

train_losses = []   # for the loss curve later

for epoch in range(epochs):

    # ---- TODO 2: fill in the five steps (see the hint table) ----
    # 1) forward pass
    # y_pred = ...

    # 2) loss (remember .view(-1, 1) on the target!)
    # loss = ...

    # 3) clear old gradients  (why before backward? -> accumulation)
    # ...

    # 4) backward pass — populate .grad on every parameter
    # ...

    # 5) update the parameters
    # ...

    raise NotImplementedError("Complete TODO 2: the five training-loop steps, then delete this line")

    train_losses.append(loss.item())
    print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

### Peek at a real gradient (run after TODO 2 works)

In the autograd lab you printed `w.grad` for a single scalar weight. Here's the same idea, but `.grad`
is now a whole `(1, 30)` gradient vector — one number per input feature — plus the bias gradient.
This is exactly what `optimizer.step()` just consumed.

In [ ]:
print("weight grad shape:", model.linear.weight.grad.shape)   # (1, 30)
print("bias grad:", model.linear.bias.grad)                    # (1,)

---
## Step 5: Evaluation — `TODO 3`: run the forward pass with **no** graph

> ### 💡 Hint — `torch.no_grad()`
> At evaluation you only need predictions, never a backward pass. Wrap the forward pass in
> `with torch.no_grad():` so PyTorch skips building the graph — saving memory/compute and signalling
> intent. (This is option 3 from the "disabling gradient tracking" section of the autograd lab.)
>
> Inside the block:
> 1. `y_pred = model(X_test_tensor)` — probabilities
> 2. threshold at 0.5 → class labels: `(y_pred > 0.5).float()`
> 3. accuracy = mean of `(y_pred_labels == y_test_tensor.view(-1, 1))` cast to float
>    — note the same `.view(-1, 1)` shape trap as the loss.

In [ ]:
# ---- TODO 3: evaluate under torch.no_grad() ----
# with torch.no_grad():
#     y_pred = ...
#     y_pred_labels = ...
#     accuracy = ...
#     print(f'Accuracy: {accuracy.item()}')
raise NotImplementedError("Complete TODO 3: evaluation under torch.no_grad()")

### Confirm the graph really was turned off

Contrast with the `grad_fn` probe from Step 3: inside `no_grad`, the output tensor has **no** `grad_fn`.

In [ ]:
with torch.no_grad():
    _p = model(X_test_tensor[:3])
print("grad_fn inside no_grad:", _p.grad_fn)   # None -> no graph was built

---
## Step 6: Visualize the loss curve (given)

A falling loss is the first sanity check that backprop is actually working.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, epochs + 1), train_losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss (training set)')
plt.title('Training Loss over Epochs')
plt.grid(True)
plt.show()

---
## Check your understanding (autograd/backprop focus)

Write a one-line answer in the cell below each — these are the concepts this lab is testing.

1. **Accumulation.** You *delete* the `optimizer.zero_grad()` line from TODO 2 and rerun. The loss
   curve looks wrong. Using Fact #3, explain in one sentence what `.grad` holds at epoch 3.
2. **Where the graph lives.** Which tensors in this notebook have `requires_grad=True` — the input
   `X_train_tensor`, or `model.linear.weight`? Why does that difference matter for what `.backward()`
   computes gradients *with respect to*?
3. **Scalar requirement.** `loss.backward()` works, but `y_pred.backward()` (before reducing to a
   scalar) would raise an error. Why? (Recall why you reduced with `.mean()` in the vector section of
   the autograd lab.)
4. **`no_grad` vs `detach()`.** Step 5 uses `torch.no_grad()`. Name one situation where you'd instead
   want `.detach()` on a single tensor while *keeping* the graph alive for everything else.

### Bonus (optional)
Swap `nn.Sigmoid()` + `nn.BCELoss()` for a single `nn.BCEWithLogitsLoss()` on the **raw** linear
output (drop the sigmoid from `forward()`). Why is fusing the sigmoid into the loss more numerically
stable when a predicted probability is very close to 0 or 1?

_Your answers:_

1. 
2. 
3. 
4. 